# Unsupervised Learning

Notebook for **Chapter 13** of the *ml* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ml/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 k-means Clustering

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(0)
C = np.array([[10, 2], [40, 3], [25, 12]]); sizes = [200, 150, 150]; sd = [[3, 0.6], [6, 0.8], [4, 2.0]]
X = np.vstack([rng.normal(C[i], sd[i], (sizes[i], 2)) for i in range(3)]); true = np.repeat([0, 1, 2], sizes)
Xs = StandardScaler().fit_transform(X)       # scale first (see below)

def kmeans(X, k, seed=0, iters=100):
    r = np.random.default_rng(seed)
    cen = [X[r.integers(len(X))]]                                   # k-means++ start
    for _ in range(k - 1):
        d2 = np.min(((X[:, None] - np.array(cen)[None])**2).sum(2), 1)
        cen.append(X[r.choice(len(X), p=d2 / d2.sum())])
    cen = np.array(cen)
    for it in range(iters):
        lab = ((X[:, None] - cen[None])**2).sum(2).argmin(1)          # assign
        new = np.array([X[lab == j].mean(0) if (lab == j).any() else cen[j] for j in range(k)])  # update
        if np.allclose(new, cen): break
        cen = new
    return lab, cen, ((X - cen[lab])**2).sum(), it + 1

lab, cen, inertia, it = kmeans(Xs, 3)
km = KMeans(3, n_init=10, random_state=0).fit(Xs)
print("scratch inertia", round(inertia, 2), "iters", it)
print("sklearn inertia", round(km.inertia_, 2), "ARI vs scratch", round(adjusted_rand_score(lab, km.labels_), 3),
      "ARI vs truth", round(adjusted_rand_score(true, km.labels_), 3))
print(StandardScaler().fit(X).inverse_transform(km.cluster_centers_).round(1).tolist())

scratch inertia 84.2 iters 2
sklearn inertia 84.2 ARI vs scratch 0.994 ARI vs truth 0.988
[[40.1, 3.0], [24.5, 11.8], [9.8, 2.0]]


In [ ]:
X2 = X.copy(); X2[:, 0] *= 1000                      # spend in rupees
for nm, Z in [("raw rupees", X2), ("scaled", StandardScaler().fit_transform(X2))]:
    m = KMeans(3, n_init=10, random_state=0).fit(Z)
    print(nm, "ARI vs truth", round(adjusted_rand_score(true, m.labels_), 3))

raw rupees ARI vs truth 0.83
scaled ARI vs truth 0.988


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
for k in range(2, 8):
    m = KMeans(k, n_init=10, random_state=0).fit(Xs)
    print(k, "inertia", round(m.inertia_, 1), "sil", round(silhouette_score(Xs, m.labels_), 3),
          "DB", round(davies_bouldin_score(Xs, m.labels_), 3), "CH", round(calinski_harabasz_score(Xs, m.labels_)))

2 inertia 476.5 sil 0.571 DB 0.691 CH 547
3 inertia 84.2 sil 0.778 DB 0.337 CH 2703
4 inertia 64.2 sil 0.695 DB 0.497 CH 2411
5 inertia 49.3 sil 0.587 DB 0.753 CH 2386
6 inertia 42.7 sil 0.415 DB 0.89 CH 2214
7 inertia 37.1 sil 0.402 DB 0.923 CH 2131


In [ ]:
def stab(Z, k, B=20):
    base = KMeans(k, n_init=10, random_state=0).fit(Z); sc = []
    for b in range(B):
        i = np.random.default_rng(b).integers(0, len(Z), len(Z))
        m = KMeans(k, n_init=10, random_state=b).fit(Z[i]); sc.append(adjusted_rand_score(base.labels_, m.predict(Z)))
    return np.mean(sc)
for k in [2, 3, 4, 6]: print("stability k", k, round(stab(Xs, k), 3))

stability k 2 1.0
stability k 3 0.996
stability k 4 0.963
stability k 6 0.876


In [ ]:
res = [KMeans(3, init='random', n_init=1, random_state=s).fit(Xs).inertia_ for s in range(20)]
print("random init single runs inertia min/max", round(min(res), 1), round(max(res), 1), "distinct", len(set(np.round(res, 1))))

random init single runs inertia min/max 84.2 584.4 distinct 5


## 13.2 Hierarchical Clustering, DBSCAN, and Distance Choices

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
P = np.array([[1, 1], [1.5, 1], [1, 2], [8, 8], [8.5, 8], [9, 9], [15, 1]])
Z = linkage(P, 'ward'); print(np.round(Z, 2)); print("cut 3 clusters", fcluster(Z, 3, 'maxclust'))

[[ 0.    1.    0.5   2.  ]
 [ 3.    4.    0.5   2.  ]
 [ 2.    7.    1.19  3.  ]
 [ 5.    8.    1.44  3.  ]
 [ 6.   10.   12.    4.  ]
 [ 9.   11.   19.15  7.  ]]
cut 3 clusters [1 1 1 2 2 2 3]


In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors
M, mt = make_moons(500, noise=0.07, random_state=0)
print("kmeans moons ARI", round(adjusted_rand_score(mt, KMeans(2, n_init=10, random_state=0).fit_predict(M)), 3))
print("DBSCAN moons ARI", round(adjusted_rand_score(mt, DBSCAN(eps=0.2, min_samples=5).fit_predict(M)), 3))

kmeans moons ARI 0.245
DBSCAN moons ARI 1.0


In [ ]:
rng2 = np.random.default_rng(0)
Mn = np.vstack([M, rng2.uniform(-1.5, 2.5, (40, 2))])
dk = np.sort(NearestNeighbors(n_neighbors=5).fit(Mn).kneighbors(Mn)[0][:, 4])
print("5th-NN dist percentiles 50/90/95/99", np.percentile(dk, [50, 90, 95, 99]).round(3))
for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    l = DBSCAN(eps=eps, min_samples=5).fit_predict(Mn)
    print("eps", eps, "clusters", len(set(l)) - (1 if -1 in l else 0), "noise", int((l == -1).sum()))

5th-NN dist percentiles 50/90/95/99 [0.064 0.12  0.25  0.884]
eps 0.05 clusters 30 noise 291
eps 0.1 clusters 3 noise 49
eps 0.15 clusters 2 noise 31
eps 0.2 clusters 2 noise 27
eps 0.3 clusters 2 noise 20
eps 0.5 clusters 1 noise 11


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
a = np.array([1, 2, 3.]); b = np.array([10, 20, 30.]); c = np.array([3, 2, 1.])
print("cos(a,b)", cosine_similarity([a], [b])[0, 0].round(3), "eucl(a,b)", euclidean_distances([a], [b])[0, 0].round(2),
      "cos(a,c)", cosine_similarity([a], [c])[0, 0].round(3), "eucl(a,c)", euclidean_distances([a], [c])[0, 0].round(2))

cos(a,b) 1.0 eucl(a,b) 33.67 cos(a,c) 0.714 eucl(a,c) 2.83


## 13.3 Gaussian Mixtures and the EM Algorithm

In [ ]:
from scipy.stats import norm
r = np.random.default_rng(3); x = np.r_[r.normal(0, 1, 60), r.normal(5, 1, 40)]
def em(mu, s, w, iters=30, show=(1, 2, 3, 5, 10, 30)):
    mu, s, w = map(np.array, (mu, s, w))
    for it in range(1, iters + 1):
        p = w * norm.pdf(x[:, None], mu, s); resp = p / p.sum(1, keepdims=True)          # E-step
        ll = np.log(p.sum(1)).sum(); Nk = resp.sum(0)
        w = Nk / len(x); mu = (resp * x[:, None]).sum(0) / Nk
        s = np.sqrt((resp * (x[:, None] - mu)**2).sum(0) / Nk)                           # M-step
        if it in show: print("it", it, "loglik", ll.round(2), "mu", mu.round(3), "sd", s.round(3), "w", w.round(3))
em([1.0, 4.0], [2.0, 2.0], [0.5, 0.5])

it 1 loglik -237.49 mu [0.383 4.042] sd [1.738 2.08 ] w [0.575 0.425]
it 2 loglik -230.34 mu [0.161 4.308] sd [1.424 1.856] w [0.571 0.429]
it 3 loglik -223.39 mu [-0.024 4.544] sd [1.133 1.528] w [0.571 0.429]
it 5 loglik -211.57 mu [-0.092 4.764] sd [0.977 1.151] w [0.582 0.418]
it 10 loglik -210.94 mu [-0.073 4.808] sd [0.993 1.098] w [0.588 0.412]
it 30 loglik -210.94 mu [-0.072 4.808] sd [0.993 1.098] w [0.588 0.412]


In [ ]:
from sklearn.mixture import GaussianMixture
G = [GaussianMixture(k, n_init=5, random_state=0).fit(Xs) for k in range(1, 7)]
for k, g in enumerate(G, 1): print("GMM k", k, "BIC", round(g.bic(Xs), 1), "AIC", round(g.aic(Xs), 1))
g = G[2]; pr = g.predict_proba(Xs)
print("weights", g.weights_.round(3), "ARI", round(adjusted_rand_score(true, g.predict(Xs)), 3))
print("rows with max prob < 0.9:", int((pr.max(1) < 0.9).sum()), "of", len(Xs))

GMM k 1 BIC 2857.7 AIC 2836.7
GMM k 2 BIC 1662.0 AIC 1615.7
GMM k 3 BIC 1220.7 AIC 1149.0
GMM k 4 BIC 1246.6 AIC 1149.6
GMM k 5 BIC 1280.2 AIC 1158.0
GMM k 6 BIC 1313.5 AIC 1166.0
weights [0.298 0.3 0.402] ARI 0.994
rows with max prob < 0.9: 0 of 500


## 13.4 PCA and Related Methods

In [ ]:
P = np.array([[2, 1], [4, 3], [6, 2], [8, 6.]]); print(P.mean(0)); C = np.cov(P.T); print(C)
v, V = np.linalg.eigh(C); print(v, V)

[5. 3.]
[[6.66666667 4.66666667]
 [4.66666667 4.66666667]]
[ 0.89405965 10.43927369] [[ 0.62867755 -0.77766609]
 [-0.77766609 -0.62867755]]


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
D = load_digits(); Xd = D.data; yd = D.target; Xc = Xd - Xd.mean(0)
vals, vecs = np.linalg.eigh(np.cov(Xc.T)); vals = vals[::-1]
p = PCA().fit(Xd)
print("eig", vals[:3].round(2), "sklearn", p.explained_variance_[:3].round(2))
U, S, Vt = np.linalg.svd(Xc, full_matrices=False); print("SVD", (S[:3]**2 / (len(Xd) - 1)).round(2))
cum = np.cumsum(p.explained_variance_ratio_)
print("cum var 2,10,20,30:", cum[[1, 9, 19, 29]].round(3), "| comps for 90%:", int(np.searchsorted(cum, 0.90)) + 1, "95%:", int(np.searchsorted(cum, 0.95)) + 1)
for k in [2, 10, 30]:
    pk = PCA(k).fit(Xd); rec = pk.inverse_transform(pk.transform(Xd)); print("k", k, "recon MSE", ((Xd - rec)**2).mean().round(3))
print("total variance", Xd.var(0, ddof=1).sum().round(1))

eig [179.01 163.72 141.79] sklearn [179.01 163.72 141.79]
SVD [179.01 163.72 141.79]
cum var 2,10,20,30: [0.285 0.738 0.894 0.959] | comps for 90%: 21 95%: 29
k 2 recon MSE 13.421
k 10 recon MSE 4.914
k 30 recon MSE 0.768
total variance 1202.1


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
for k in [2, 10, 30, 64]:
    print("logreg on", k, "PCs CV acc", cross_val_score(make_pipeline(PCA(k), LogisticRegression(max_iter=3000)), Xd, yd, cv=5).mean().round(3))

logreg on 2 PCs CV acc 0.574
logreg on 10 PCs CV acc 0.888
logreg on 30 PCs CV acc 0.91
logreg on 64 PCs CV acc 0.913


## 13.5 t-SNE and UMAP: Pictures of High-Dimensional Data

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
kd = KMeans(10, n_init=10, random_state=0).fit(Xd)
print("k-means on 64 pixels: ARI", round(adjusted_rand_score(yd, kd.labels_), 3), "NMI", round(normalized_mutual_info_score(yd, kd.labels_), 3),
      "silhouette", round(silhouette_score(Xd, kd.labels_), 3))
for perp in [5, 30, 100]:
    Y = TSNE(2, perplexity=perp, random_state=0, init='pca').fit_transform(Xd)
    print("tsne perplexity", perp, "k-means-on-2D ARI", round(adjusted_rand_score(yd, KMeans(10, n_init=10, random_state=0).fit_predict(Y)), 3))

k-means on 64 pixels: ARI 0.666 NMI 0.742 silhouette 0.183
tsne perplexity 5 k-means-on-2D ARI 0.823
tsne perplexity 30 k-means-on-2D ARI 0.889
tsne perplexity 100 k-means-on-2D ARI 0.839


## 13.6 Association Rules and Density Estimation

In [ ]:
import itertools, pandas as pd
baskets = [{'bread','butter','milk'}, {'bread','butter'}, {'bread','milk'}, {'milk','eggs'},
           {'bread','butter','eggs'}, {'bread','milk','eggs'}, {'milk'}, {'bread','butter','milk','eggs'},
           {'bread'}, {'milk','butter'}]
N = len(baskets); items = sorted(set().union(*baskets))
def sup(s): return sum(set(s) <= b for b in baskets) / N
print("N", N, {i: sup([i]) for i in items})
rows = []
for a, b in itertools.permutations(items, 2):
    s = sup([a, b])
    if s >= 0.3: rows.append((a, b, s, s / sup([a]), s / sup([a]) / sup([b])))
print(pd.DataFrame(rows, columns=['ante', 'cons', 'support', 'conf', 'lift']).sort_values('lift', ascending=False).round(3).to_string(index=False))

N 10 {'bread': 0.7, 'butter': 0.5, 'eggs': 0.4, 'milk': 0.7}
  ante   cons  support  conf  lift
 bread butter      0.4 0.571 1.143
butter  bread      0.4 0.800 1.143
 bread   eggs      0.3 0.429 1.071
  eggs  bread      0.3 0.750 1.071
  eggs   milk      0.3 0.429 1.071
  milk   eggs      0.3 0.429 1.071
butter   milk      0.3 0.600 0.857
  milk butter      0.3 0.429 0.857
 bread   milk      0.4 0.571 0.816
  milk   bread      0.4 0.571 0.816


In [ ]:
from scipy.stats import gaussian_kde
r = np.random.default_rng(0); v = np.r_[r.normal(0, 1, 300), r.normal(5, 1, 200)]
print("kde density at 0, 2.5, 5:", gaussian_kde(v)([0, 2.5, 5]).round(4))

kde density at 0, 2.5, 5: [0.1895 0.0444 0.1224]


## 13.7 Beyond the Basics: HDBSCAN, NMF, ICA and Topic Models

Section 13.2 ended with DBSCAN's weak spot: one `eps` for the whole data set. Section 13.4 left PCA as the only "find directions" tool. This section adds four methods that fix exactly those limits. Each answers a different question: **HDBSCAN** (clusters of different density), **NMF** (parts that add up), **ICA** (unmix independent sources), **LDA** (topics in text). All outputs are real, run with scikit-learn 1.9.1.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
rng = np.random.default_rng(0)
X1, y1 = make_blobs(n_samples=[200, 200], centers=[[0, 0], [6, 0]], cluster_std=[0.3, 1.5], random_state=0)
X2 = rng.uniform(-3, 9, (40, 2)); X = np.r_[X1, X2]; y = np.r_[y1, -np.ones(40, int)]
for eps in (0.3, 0.6, 1.0):
    lab = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    print("DBSCAN eps", eps, "clusters", len(set(lab) - {-1}), "noise", (lab == -1).sum())
h = HDBSCAN(min_cluster_size=30, copy=True).fit(X)
print("HDBSCAN clusters", len(set(h.labels_) - {-1}), "noise", (h.labels_ == -1).sum(),
      "ARI on non-noise truth", round(adjusted_rand_score(y[y >= 0], h.labels_[y >= 0]), 3))

DBSCAN eps 0.3 clusters 1 noise 243
DBSCAN eps 0.6 clusters 3 noise 113
DBSCAN eps 1.0 clusters 2 noise 38
HDBSCAN clusters 2 noise 63 ARI on non-noise truth 0.858


In [ ]:
import numpy as np
from sklearn.decomposition import NMF, FastICA, PCA, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
rng = np.random.default_rng(0)
t = np.linspace(0, 8, 2000)
S = np.c_[np.sign(np.sin(3 * t)), np.sin(5 * t + 1)]
A = np.array([[1.0, 0.5], [0.4, 1.0]]); Xm = S @ A.T + 0.02 * rng.normal(size=S.shape)
ica = FastICA(n_components=2, random_state=0, whiten="unit-variance").fit(Xm)
est = ica.transform(Xm)
c = np.abs(np.corrcoef(S.T, est.T)[:2, 2:]).round(3)
p = PCA(2).fit_transform(Xm); cp = np.abs(np.corrcoef(S.T, p.T)[:2, 2:]).round(3)
print("ICA |corr| true vs recovered:\n", c, "\nPCA |corr|:\n", cp)
# NMF on parts
V = np.array([[5,4,0,0],[4,5,1,0],[0,1,5,4],[0,0,4,5],[3,3,3,3]], float)
m = NMF(2, init="nndsvda", random_state=0, max_iter=1000).fit(V)
print("W\n", m.transform(V).round(2), "\nH\n", m.components_.round(2), "\nerr", round(m.reconstruction_err_, 3))
docs = ["goal match team win league", "team coach win season goal", "match league season coach team",
        "stock market price trade bank", "bank price market loan trade", "loan stock bank market invest",
        "goal team bank market"] 
cv = CountVectorizer(); D = cv.fit_transform(docs)
lda = LatentDirichletAllocation(2, random_state=0, max_iter=200).fit(D)
w = np.array(cv.get_feature_names_out())
for k, row in enumerate(lda.components_): print("topic", k, w[row.argsort()[::-1][:4]])
print(lda.transform(D).round(2))

ICA |corr| true vs recovered:
 [[0.998 0.059]
 [0.03  0.999]] 
PCA |corr|:
 [[0.82  0.572]
 [0.497 0.867]]
W
 [[0.   1.61]
 [0.04 1.61]
 [0.45 0.13]
 [0.45 0.  ]
 [0.29 1.04]] 
H
 [[ 0.    0.7   9.96 10.05]
 [ 2.8   2.78  0.19  0.  ]] 
err 1.63
topic 0 ['team' 'goal' 'win' 'season']
topic 1 ['market' 'bank' 'trade' 'price']
[[0.92 0.08]
 [0.92 0.08]
 [0.92 0.08]
 [0.08 0.92]
 [0.08 0.92]
 [0.09 0.91]
 [0.5  0.5 ]]


## Chapter recap

## Practice set

Problem 13.P.1 (easy) Points 0, 2, 10, 12 with k = 2 and starting centroids 0 and 12. Give the final centroids and inertia.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ml/ch13.html